# Chapter 13: Sorting and Searching Algorithms

Organise collections, retrieve values, and rank illustrative AI results. Sorting also supports graph algorithms such as ordering Kruskal’s edges by weight.

**Prerequisites:** JavaScript arrays, loops, functions, and basic recursion.

Select **JavaScript (Node.js)** and run cells from top to bottom. All examples use standard JavaScript with no external packages. Every one of the supplied guide’s 12 “Try it” tasks has a runnable worked solution with assertions. Restart the kernel before running all cells again if declarations already exist.

## 1. Define what comes first

A comparator returns a negative number when a comes before b, zero for equal sort keys, and a positive number when a comes after b. All comparison sorts below use this sign convention. Check time, extra space (including output and recursion), stability (relative order of equal-key records), and mutation when choosing an algorithm.

Bubble sort compares adjacent values and moves the largest remaining value to the right on each pass. On [5, 4, 3, 2, 1], its first pass produces [4, 3, 2, 1, 5].

In [ ]:
const ascending = (a, b) => a - b;
function swap(array, a, b) {
   [array[a], array[b]] = [array[b], array[a]];
 }

### Try it 1

Why return zero for equal keys? What is the array after the next complete bubble pass?

Predict the answer before running the solution below.

**Worked solution 1.** Zero represents equality, allowing stable algorithms to retain ties. The second pass produces [3, 2, 1, 4, 5].

In [ ]:
(() => {
const assert = require("node:assert/strict");
const values = [5, 4, 3, 2, 1];
const passes = [];
for (let end = values.length - 1; end >= values.length - 2; end--) {
 for (let j = 0; j < end; j++) if (ascending(values[j], values[j + 1]) > 0) swap(values, j, j + 1);
 passes.push(values.slice());
}
assert.equal(ascending(4, 4), 0);
assert.deepEqual(passes, [[4, 3, 2, 1, 5], [3, 2, 1, 4, 5]]);
console.log('First and second passes:', passes);
})();

## 2. Compare simple sorting methods

Bubble sort swaps neighbors, selection sort swaps the minimum of each remaining suffix into place, and insertion sort shifts larger values to insert into a sorted prefix.

All three mutate the array, use O(1) extra space, and take O(n²) worst-case time. Early-exit bubble sort and insertion sort take O(n) on sorted input. Selection sort still makes O(n²) comparisons. The shown bubble and insertion sorts are stable; selection sort is not generally stable.

In [ ]:
function bubbleSort(a, cmp = ascending) {
   for (let end = a.length - 1; end > 0; end--) {
     let changed = false;
     for (let j = 0; j < end; j++) {
       if (cmp(a[j], a[j + 1]) > 0) {
         swap(a, j, j + 1);
         changed = true;
       }
     }
     if (!changed) break;
   }
   return a;
 }

In [ ]:
function selectionSort(a, cmp = ascending) {
   for (let i = 0; i < a.length - 1; i++) {
     let min = i;
     for (let j = i + 1; j < a.length; j++)
       if (cmp(a[j], a[min]) < 0) min = j;
     if (min !== i) swap(a, i, min);
   }
   return a;
 }

In [ ]:
function insertionSort(a, cmp = ascending) {
   for (let i = 1; i < a.length; i++) {
     const value = a[i];
     let j = i;
     while (j > 0 && cmp(a[j - 1], value) > 0) {
       a[j] = a[j - 1];
       j--;
     }
     a[j] = value;
   }
   return a;
 }

In [ ]:
for (const sort of [bubbleSort, selectionSort, insertionSort]) console.log(sort.name, sort([5, 4, 3, 2, 1]));

### Try it 2

Why does insertion sort do little work on nearly sorted input? How can selection sort reorder equal keys?

Predict the answer before running the solution below.

**Worked solution 2.** Few inversions require few shifts. Selection sort can swap an equal-key record across another: [2a, 2b, 1] becomes [1, 2b, 2a].

In [ ]:
(() => {
const assert = require("node:assert/strict");
const records = [{ id: '2a', key: 2 }, { id: '2b', key: 2 }, { id: '1', key: 1 }];
const cmp = (a, b) => a.key - b.key;
assert.deepEqual(selectionSort(records.slice(), cmp).map(x => x.id), ['1', '2b', '2a']);
assert.deepEqual(insertionSort(records.slice(), cmp).map(x => x.id), ['1', '2a', '2b']);
let comparisons = 0;
assert.deepEqual(insertionSort([1, 2, 3, 5, 4], (a, b) => { comparisons++; return a - b; }), [1, 2, 3, 4, 5]);
assert.equal(comparisons, 5);
console.log('One inversion needs one shift; this example makes', comparisons, 'comparisons.');
console.log('Selection reorders ties:', selectionSort(records.slice(), cmp));
})();

## 3. Move values farther with Shell sort

Shell sort performs insertion sort on interleaved subsequences separated by a gap, then reduces the gap to 1. Sorting a larger-gap subsequence alone does not sort the whole array.

This halving-gap version mutates its input, uses O(1) extra space, takes O(n²) worst-case time, and is not generally stable. Its best case is O(n log n). Bounds depend on the gap sequence.

In [ ]:
function shellSort(a, cmp = ascending) {
   for (let gap = Math.floor(a.length / 2);
        gap > 0; gap = Math.floor(gap / 2)) {
     for (let i = gap; i < a.length; i++) {
       const value = a[i];
       let j = i;
       while (j >= gap && cmp(a[j - gap], value) > 0) {
         a[j] = a[j - gap];
         j -= gap;
       }
       a[j] = value;
     }
   }
   return a;
 }

### Try it 3

For length 10, list the gaps. Why is stopping at gap 2 insufficient?

Predict the answer before running the solution below.

**Worked solution 3.** The gaps are 5, 2, 1. Gap 2 only sorts even and odd index subsequences separately.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const gaps = [];
for (let gap = Math.floor(10 / 2); gap > 0; gap = Math.floor(gap / 2)) gaps.push(gap);
assert.deepEqual(gaps, [5, 2, 1]);
const incomplete = [2, 1, 4, 3]; // Even positions 2,4 and odd positions 1,3 are individually sorted.
assert.ok(incomplete[0] > incomplete[1]);
assert.deepEqual(shellSort(incomplete.slice()), [1, 2, 3, 4]);
console.log({ gaps, counterexampleAfterGap2: incomplete });
})();

## 4. Divide and merge sorted sequences

Merge sort divides into halves, recursively sorts them, and merges their front values. Taking the left record on a tie preserves stability. There are O(log n) levels, each processing O(n) values: O(n log n) time in all cases for nontrivial inputs.

This version returns a new array, with O(n) peak auxiliary storage plus an O(log n) call stack. Objects are copied by reference, not cloned.

In [ ]:
function mergeSort(a, cmp = ascending) {
   if (a.length <= 1) return [...a];
   const middle = Math.floor(a.length / 2);
   const left = mergeSort(a.slice(0, middle), cmp);
   const right = mergeSort(a.slice(middle), cmp);
   const result = [];
   let i = 0, j = 0;
   while (i < left.length && j < right.length) {
     if (cmp(left[i], right[j]) <= 0) result.push(left[i++]);
     else result.push(right[j++]);
   }
   while (i < left.length) result.push(left[i++]);
   while (j < right.length) result.push(right[j++]);
   return result;
 }

In [ ]:
console.log(mergeSort([4, 1, 3, 2])); // [1, 2, 3, 4]

### Try it 4

Merge [1, 4, 7] and [2, 4, 6]. Which record with key 4 comes first?

Predict the answer before running the solution below.

**Worked solution 4.** The merged keys are [1, 2, 4, 4, 6, 7]. The left-hand 4 is taken first to preserve the original order across the split.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const left = [{ key: 1, id: 'L1' }, { key: 4, id: 'L4' }, { key: 7, id: 'L7' }];
const right = [{ key: 2, id: 'R2' }, { key: 4, id: 'R4' }, { key: 6, id: 'R6' }];
const merged = [];
let i = 0, j = 0;
while (i < left.length && j < right.length) {
 if (left[i].key <= right[j].key) merged.push(left[i++]);
 else merged.push(right[j++]);
}
merged.push(...left.slice(i), ...right.slice(j));
assert.deepEqual(merged.map(x => x.key), [1, 2, 4, 4, 6, 7]);
assert.deepEqual(merged.filter(x => x.key === 4).map(x => x.id), ['L4', 'R4']);
console.log(merged);
})();

## 5. Partition around a quick sort pivot

The final value is the pivot. A boundary separates values smaller than the pivot; after partitioning, equal or larger values lie on its right. The regions still require recursive sorting.

Balanced partitions give O(n log n) time. Random permutations of distinct keys give O(n log n) average time. Sorted input and many equal keys cause O(n²) worst-case time and O(n) stack depth for this version. It mutates its input and is not stable. Randomized pivots, three-way partitioning, and stack safeguards address common weaknesses.

In [ ]:
function quickSort(a, cmp = ascending) {
   function sort(low, high) {
     if (low >= high) return;
     const pivot = a[high];
     let boundary = low;
     for (let j = low; j < high; j++) {
       if (cmp(a[j], pivot) < 0) {
         swap(a, boundary, j);
         boundary++;
       }
     }
     swap(a, boundary, high);
     sort(low, boundary - 1);
     sort(boundary + 1, high);
   }
   sort(0, a.length - 1);
   return a;
 }

### Try it 5

Which pivot is chosen for [1, 2, 3, 4]? How many values lie on either side?

Predict the answer before running the solution below.

**Worked solution 5.** The pivot is 4, with three values on its left and none on its right. Repetition of this split makes recursion unbalanced.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const values = [1, 2, 3, 4];
const pivot = values.at(-1);
const left = values.slice(0, -1).filter(x => x < pivot);
const right = values.slice(0, -1).filter(x => x >= pivot);
assert.equal(pivot, 4);
assert.equal(left.length, 3);
assert.equal(right.length, 0);
console.log({ pivot, left, right });
})();

## 6. Count values in a small integer range

Counting sort counts occurrences, using an offset for negative integers. It returns a new array of numbers. For range k = max − min + 1, time and total extra space are O(n + k); the counting array alone uses O(k).

Only safe integers in a manageable range are suitable. A huge range can fail allocation or waste memory even for a tiny input. This reconstruction of bare numbers does not demonstrate stable record sorting; records require cumulative counts and stable placement.

In [ ]:
function countingSort(a) {
   if (a.length === 0) return [];
   let min = a[0], max = a[0];
   for (const value of a) {
     if (!Number.isSafeInteger(value))
       throw new Error("Use safe integers");
     min = Math.min(min, value);
     max = Math.max(max, value);
   }
   const counts = Array(max - min + 1).fill(0);
   for (const value of a) counts[value - min]++;
   const result = [];
   for (let offset = 0; offset < counts.length; offset++) {
     for (let count = counts[offset]; count > 0; count--)
       result.push(offset + min);
   }
   return result;
 }

In [ ]:
console.log(countingSort([3, -1, 2, 3, 0])); // [-1, 0, 2, 3, 3]

### Try it 6

Is counting sort suitable for [1, 1000000000]? Explain with n and k.

Predict the answer before running the solution below.

**Worked solution 6.** No: n = 2 but k = 1,000,000,000. Do not allocate or scan a billion counts for two values.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const values = [1, 1_000_000_000];
const n = values.length, k = Math.max(...values) - Math.min(...values) + 1;
assert.equal(n, 2);
assert.equal(k, 1_000_000_000);
console.log({ n, k, suitable: false }); // Calculate the range without allocating it.
assert.deepEqual(mergeSort(values), values);
})();

## 7. Distribute numeric values into buckets

Bucket sort groups numeric intervals, sorts each with insertion sort, then concatenates in interval order. Width is interval width, not bucket count. It returns a new array. Inputs must be finite and bucket count manageable.

For b buckets of occupancies mᵢ, work is bounded by O(n + b + Σmᵢ²); worst case is O(n + b + n²), and extra space O(n + b). Expected O(n) requires suitable distribution and O(n) buckets with bounded expected occupancy. Tiny width across a huge range creates many empty buckets.

In [ ]:
function bucketSort(a, width = 5) {
   if (!(width > 0) || !Number.isFinite(width))
     throw new Error("Use a positive finite bucket width");
   if (a.length === 0) return [];
   let min = a[0], max = a[0];
   for (const value of a) {
     if (!Number.isFinite(value)) throw new Error("Finite values only");
     min = Math.min(min, value);
     max = Math.max(max, value);
   }
   const count = Math.floor((max - min) / width) + 1;
   const buckets = Array.from({ length: count }, () => []);
   for (const value of a)
     buckets[Math.floor((value - min) / width)].push(value);
   const result = [];
   for (const bucket of buckets) {
     insertionSort(bucket);
     for (const value of bucket) result.push(value);
   }
   return result;
 }

In [ ]:
console.log(bucketSort([12, 3, 9, 1, 7], 5)); // [1, 3, 7, 9, 12]

### Try it 7

With minimum 1 and width 5, where does 6 go? Why not [1, 6)?

Predict the answer before running the solution below.

**Worked solution 7.** It goes into bucket index 1, the second interval [6, 11). The right endpoint of [1, 6) is excluded.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const min = 1, width = 5, value = 6;
const index = Math.floor((value - min) / width);
assert.equal(index, 1);
assert.ok(value >= min + index * width && value < min + (index + 1) * width);
console.log({ value, bucketIndex: index, interval: '[6, 11)' });
})();

## 8. Sort integer digits with radix sort

Least-significant-digit radix sort processes ones, tens, hundreds, and so on. Each pass must be stable to preserve earlier digit ordering. Appending in encounter order and reading buckets from 0 to 9 makes each pass stable.

This version returns a new array and supports nonnegative safe integers in base 10. With d digits and base b, time is O(d(n + b)) and peak auxiliary space O(n + b). Missing leading digits are zero.

In [ ]:
function radixSort(a) {
   let max = 0;
   for (const value of a) {
     if (!Number.isSafeInteger(value) || value < 0)
       throw new Error("Use nonnegative safe integers");
     max = Math.max(max, value);
   }
   let result = [...a];
   for (let place = 1; place <= max; place *= 10) {
     const buckets = Array.from({ length: 10 }, () => []);
     for (const value of result) {
       const digit = Math.floor(value / place) % 10;
       buckets[digit].push(value);
     }
     result = buckets.flat();
   }
   return result;
 }

In [ ]:
const radixExample = [170, 45, 75, 90, 802, 24, 2, 66];
console.log(radixSort(radixExample));

### Try it 8

Why does 170 precede 90 after the ones pass? What if a later pass reverses equal-digit items?

Predict the answer before running the solution below.

**Worked solution 8.** Both have ones digit 0 and 170 appeared first. A later unstable pass can destroy ordering established by an earlier pass.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const buckets = Array.from({ length: 10 }, () => []);
for (const value of radixExample) buckets[value % 10].push(value);
const ones = buckets.flat();
assert.deepEqual(ones, [170, 90, 802, 2, 24, 45, 75, 66]);
// 11 and 12 share the tens digit; reversing that bucket destroys ones ordering.
const unstableTensResult = [11, 12].reverse();
assert.deepEqual(unstableTensResult, [12, 11]);
assert.deepEqual(radixSort([12, 11]), [11, 12]);
console.log('Stable ones pass:', ones, 'Unstable tens counterexample:', unstableTensResult);
})();

## 9. Search sequentially or halve the range

Linear search works on unsorted data, taking O(n) worst-case time. Binary search requires input sorted under the same comparator and takes O(log n) worst-case time. Both use O(1) extra space.

Binary search returns an index in its sorted input, not the earlier unsorted array. Duplicate keys may yield any matching index. Sorting for one lookup can cost more than linear search; sorting once can pay off across repeated queries.

In [ ]:
function linearSearch(a, target, equals = (x, y) => x === y) {
   for (let i = 0; i < a.length; i++)
     if (equals(a[i], target)) return i;
   return -1;
 }
function binarySearch(a, target, cmp = ascending) {
   let low = 0, high = a.length - 1;
   while (low <= high) {
     const mid = low + Math.floor((high - low) / 2);
     const comparison = cmp(a[mid], target);
     if (comparison < 0) low = mid + 1;
     else if (comparison > 0) high = mid - 1;
     else return mid;
   }
   return -1;
 }

In [ ]:
console.log(linearSearch([8, 2, 5], 2)); // 1
console.log(binarySearch([2, 5, 8, 11, 14, 17, 20], 14)); // 4

### Try it 9

What does binarySearch([2, 5, 8], 6) return? Why must comparison rules match?

Predict the answer before running the solution below.

**Worked solution 9.** It returns −1. If sort and search disagree, the search may discard the half containing the target.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.equal(binarySearch([2, 5, 8], 6), -1);
const descending = (a, b) => b - a;
const values = mergeSort([2, 5, 8], descending);
assert.equal(binarySearch(values, 8, descending), 0);
assert.equal(binarySearch(values, 8), -1); // Wrong comparator for this order.
console.log('Missing target:', -1, 'Descending search with matching comparator:', binarySearch(values, 8, descending));
})();

## 10. Estimate a position with interpolation

Interpolation search estimates position from endpoint values in sorted ascending numeric data. With suitable uniform-distribution assumptions, average time is O(log log n); worst-case time is O(n). Extra space is O(1). Binary search keeps an O(log n) worst-case bound without those distribution assumptions.

Assume finite values and target, with subtraction representable in the numeric range. Equal endpoints require a direct equality check to avoid division by zero. This teaching implementation is not a general floating-point search routine.

In [ ]:
function interpolationSearch(a, target) {
   let low = 0, high = a.length - 1;
   while (low <= high && target >= a[low] && target <= a[high]) {
     if (a[low] === a[high])
       return a[low] === target ? low : -1;
     const ratio = (target - a[low]) / (a[high] - a[low]);
     const estimate = low + Math.floor((high - low) * ratio);
     const position = Math.max(low, Math.min(high, estimate));
     if (a[position] === target) return position;
     if (a[position] < target) low = position + 1;
     else high = position - 1;
   }
   return -1;
 }

In [ ]:
console.log(interpolationSearch([10, 20, 30, 40, 50], 40)); // 3
console.log(interpolationSearch([7, 7, 7], 7)); // 0

### Try it 10

Why is interpolation not automatically better for [1, 2, 3, 4, 1000000]? What about [7, 7, 7]?

Predict the answer before running the solution below.

**Worked solution 10.** The outlier makes endpoint estimates poor. Equal endpoints require checking equality before dividing by their difference.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const values = [1, 2, 3, 4, 1_000_000];
const ratio = (4 - values[0]) / (values.at(-1) - values[0]);
const firstEstimate = Math.floor((values.length - 1) * ratio);
assert.equal(firstEstimate, 0); // Target actually lies at index 3.
assert.equal(interpolationSearch(values, 4), 3);
assert.equal(interpolationSearch([7, 7, 7], 7), 0);
assert.equal(interpolationSearch([7, 7, 7], 8), -1);
console.log({ firstEstimate, actualIndex: 3, equalEndpointsResult: 0 });
})();

## 11. Shuffle with Fisher–Yates

Visit positions from right to left. At i, choose j uniformly from 0 through i, including i, and swap. Independent uniform choices give every permutation probability 1/n!. The algorithm mutates the array, takes O(n) time, and uses O(1) extra space.

The supplied random generator must return values in [0, 1). Math.random offers neither seeded reproducibility nor cryptographic guarantees. Sorting with a random comparator is inconsistent and does not produce a uniform shuffle.

In [ ]:
function shuffle(a, random = Math.random) {
   for (let i = a.length - 1; i > 0; i--) {
     const j = Math.floor(random() * (i + 1));
     swap(a, i, j);
   }
   return a;
 }

In [ ]:
const cards = [1, 2, 3, 4];
console.log('One random shuffle:', shuffle(cards));

### Try it 11

At i = 3, which j values are allowed? What if random always returns zero?

Predict the answer before running the solution below.

**Worked solution 11.** j can be 0, 1, 2, or 3. A zero generator changes [1, 2, 3, 4] to [2, 3, 4, 1]; this deterministic generator is not a fair shuffle.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const allowed = [0, 0.25, 0.5, 0.75].map(r => Math.floor(r * 4));
assert.deepEqual(allowed, [0, 1, 2, 3]);
const values = [1, 2, 3, 4];
assert.equal(shuffle(values, () => 0), values);
assert.deepEqual(values, [2, 3, 4, 1]);
console.log({ allowed, zeroGeneratorResult: values });
})();

## 12. Rank AI results and prepare data

Sort illustrative relevance scores descending, break ties deterministically by ID, and keep the first k. Sorting orders the scores; it does not assess their accuracy. Sorting all n candidates takes O(n log n); a bounded heap selects top k in O(n log k) for k ≥ 2 with O(k) heap space.

Keep input and label in the same record when shuffling before a data split. The tiny example demonstrates mechanics only. Real evaluation may need grouping, stratification, or time-based splits to avoid leakage. Use a seeded generator for repeatable experiments.

In [ ]:
const passages = [
   { id: "A", score: 0.72, text: "Account settings" },
   { id: "B", score: 0.95, text: "Password reset steps" },
   { id: "C", score: 0.81, text: "Account recovery" },
   { id: "D", score: 0.95, text: "Reset troubleshooting" }
 ];
 const compareId = (a, b) => a < b ? -1 : a > b ? 1 : 0;
 const rankedPassages = mergeSort(passages, (a, b) =>
   (b.score - a.score) || compareId(a.id, b.id));
 const bestPassages = rankedPassages.slice(0, 2);
 console.log(bestPassages.map(item => item.id)); // ["B", "D"]

// Shuffle labeled examples together before splitting.
 const examples = [
   { input: "hello", label: "greeting" },
   { input: "thanks", label: "gratitude" },
   { input: "hi", label: "greeting" },
   { input: "thank you", label: "gratitude" }
 ];
 const mixedExamples = shuffle([...examples]);
 const cut = Math.floor(mixedExamples.length * 0.75);
 const training = mixedExamples.slice(0, cut);
 const validation = mixedExamples.slice(cut);

In [ ]:
console.log('Training:', training);
console.log('Validation:', validation);

### Try it 12

If A’s score becomes 0.99, which IDs are selected? Why not shuffle inputs and labels separately?

Predict the answer before running the solution below.

**Worked solution 12.** A and B are selected. B wins the 0.95 tie with D by ID. Separate shuffles can attach the wrong label to an input.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const updated = passages.map(item => item.id === 'A' ? { ...item, score: 0.99 } : item);
const selected = mergeSort(updated, (a, b) => (b.score - a.score) || compareId(a.id, b.id)).slice(0, 2);
assert.deepEqual(selected.map(x => x.id), ['A', 'B']);
assert.equal(passages[0].score, 0.72);
const inputs = ['hello', 'thanks'];
const labels = ['greeting', 'gratitude'];
const brokenPairs = shuffle(inputs.slice(), () => 0).map((input, i) => ({ input, label: labels[i] }));
assert.deepEqual(brokenPairs[0], { input: 'thanks', label: 'greeting' });
console.log('Selected IDs:', selected.map(x => x.id), 'Separately shuffled mismatch:', brokenPairs);
console.log('Shuffle whole records to preserve input/label correspondence.');
})();

## 13. Compare algorithms before choosing

These properties describe the versions above. Extra space includes returned arrays and recursion stacks.

| Sort | Best time | Worst time | Extra space | Stable |
| --- | --- | --- | --- | --- |
| Bubble, early exit | O(n) | O(n²) | O(1) | Yes |
| Selection | O(n²) | O(n²) | O(1) | No |
| Insertion | O(n) | O(n²) | O(1) | Yes |
| Shell, halving gaps | O(n log n) | O(n²) | O(1) | No |
| Merge | O(n log n) | O(n log n) | O(n) | Yes |
| Quick, last pivot | O(n log n) | O(n²) | O(n) worst stack | No |

| Distribution sort | Time | Extra space | Restriction |
| --- | --- | --- | --- |
| Counting | O(n + k) | O(n + k) | Safe integers, manageable range k |
| Bucket | O(n + b + Σmᵢ²) bound | O(n + b) | Finite values, manageable bucket count b |
| Radix | O(d(n + b)) | O(n + b) | Nonnegative safe integers; d digits, base b |

Insertion sort suits small or nearly sorted inputs. Merge sort gives predictable time and stable ordering. Counting and radix exploit integer structure; bucket sort depends on distribution. Binary search suits repeated queries over sorted arrays, while linear search avoids preprocessing for small or unsorted collections.

Use a full sort when you need complete order, or a heap when you repeatedly need the next priority item.

## 14. Build a small project: an AI search-results viewer

Rank ten mock passages, filter by threshold, and display the best three with score descending and ID ascending as tie-break. Maintain a separate ID-sorted list for binary lookup: relevance order and ID order serve different queries.

In [ ]:
const viewerPassages = [
 { id: 'P01', score: 0.72, text: 'Account settings' },
 { id: 'P02', score: 0.95, text: 'Password reset steps' },
 { id: 'P03', score: 0.81, text: 'Account recovery' },
 { id: 'P04', score: 0.95, text: 'Reset troubleshooting' },
 { id: 'P05', score: 0.66, text: 'Email preferences' },
 { id: 'P06', score: 0.88, text: 'Recovery codes' },
 { id: 'P07', score: 0.54, text: 'Billing history' },
 { id: 'P08', score: 0.79, text: 'Two-factor authentication' },
 { id: 'P09', score: 0.61, text: 'Profile photo' },
 { id: 'P10', score: 0.92, text: 'Locked account help' }
];
const compareRelevance = (a, b) => (b.score - a.score) || compareId(a.id, b.id);
const threshold = 0.8;
const visiblePassages = mergeSort(viewerPassages, compareRelevance)
 .filter(item => item.score >= threshold).slice(0, 3);
console.table(visiblePassages);
const passagesById = mergeSort(viewerPassages, (a, b) => compareId(a.id, b.id));
function findPassageById(id) {
 const index = binarySearch(passagesById, { id }, (a, b) => compareId(a.id, b.id));
 return index === -1 ? undefined : passagesById[index];
}
console.log('Lookup P06:', findPassageById('P06'));
console.log('Missing ID:', findPassageById('P99'));

## 15. Checks and review

The following checks cover all algorithms, comparator direction, stability, mutation, empty and duplicate inputs, invalid numeric inputs, search results, deterministic shuffle mechanics, and the small project. Seeded cases make the checks repeatable; no test allocates a huge counting array or bucket range.

In [ ]:
(() => {
 const assert = require('node:assert/strict');
 const comparisonSorts = [bubbleSort, selectionSort, insertionSort, shellSort, mergeSort, quickSort];
 const cases = [[], [4], [2, 2, 2], [3, -1, 0, 3, -5], [1, 2, 3, 4], [4, 3, 2, 1]];
 let seed = 13;
 const random = () => ((seed = (Math.imul(seed, 1664525) + 1013904223) >>> 0) / 2 ** 32);
 for (let trial = 0; trial < 100; trial++) {
  cases.push(Array.from({ length: Math.floor(random() * 40) }, () => Math.floor(random() * 101) - 50));
 }
 for (const values of cases) {
  const expected = values.slice().sort(ascending);
  for (const sort of comparisonSorts) {
   const input = values.slice();
   const result = sort(input);
   assert.deepEqual(result, expected, sort.name);
   if (sort === mergeSort) { assert.notEqual(result, input); assert.deepEqual(input, values); }
   else assert.equal(result, input);
   assert.deepEqual(sort(values.slice(), (a, b) => b - a), expected.slice().reverse());
  }
  for (const sort of [countingSort, bucketSort]) {
   const input = values.slice();
   assert.deepEqual(sort(input), expected);
   assert.deepEqual(input, values);
  }
  const nonnegative = values.map(Math.abs);
  assert.deepEqual(radixSort(nonnegative), nonnegative.slice().sort(ascending));
  assert.deepEqual(nonnegative, values.map(Math.abs));
  for (const target of [-51, ...values, 51]) {
   const linear = linearSearch(values, target);
   assert.equal(linear, values.indexOf(target));
   for (const search of [binarySearch, interpolationSearch]) {
    const index = search(expected, target);
    if (expected.includes(target)) assert.equal(expected[index], target);
    else assert.equal(index, -1);
   }
  }
  const shuffled = shuffle(values.slice(), random);
  assert.deepEqual(shuffled.slice().sort(ascending), expected);
 }
 const records = [{ id: 'a', key: 2 }, { id: 'b', key: 1 }, { id: 'c', key: 2 }, { id: 'd', key: 1 }];
 for (const sort of [bubbleSort, insertionSort, mergeSort]) {
  assert.deepEqual(sort(records.slice(), (a, b) => a.key - b.key).map(x => x.id), ['b', 'd', 'a', 'c']);
 }
 // Comparator signs need not be exactly -1 and +1.
 assert.deepEqual(mergeSort([3, 1, 2], (a, b) => (a - b) * 100), [1, 2, 3]);
 assert.throws(() => countingSort([1.5]), /safe integers/);
 assert.throws(() => countingSort([NaN]), /safe integers/);
 assert.throws(() => bucketSort([1], 0), /positive finite/);
 assert.throws(() => bucketSort([Infinity]), /Finite values/);
 for (const invalid of [-1, 1.5, Infinity, Number.MAX_SAFE_INTEGER + 1]) {
  assert.throws(() => radixSort([invalid]), /nonnegative safe integers/);
 }
 assert.deepEqual(radixSort([0, 0]), [0, 0]);
 assert.deepEqual(radixSort([Number.MAX_SAFE_INTEGER, 0, 10]), [0, 10, Number.MAX_SAFE_INTEGER]);
 assert.deepEqual(bucketSort([1.25, -0.5, 1.2], 0.5), [-0.5, 1.2, 1.25]);
 assert.equal(linearSearch(records, { key: 1 }, (a, b) => a.key === b.key), 1);
 assert.deepEqual(bestPassages.map(x => x.id), ['B', 'D']);
 assert.equal(training.length, 3);
 assert.equal(validation.length, 1);
 assert.equal(new Set([...training, ...validation]).size, examples.length);
 assert.ok([...training, ...validation].every(record => examples.includes(record)));
 assert.deepEqual(visiblePassages.map(x => x.id), ['P02', 'P04', 'P10']);
 assert.equal(findPassageById('P06').text, 'Recovery codes');
 assert.equal(findPassageById('P99'), undefined);
 // Enumerate all legal choices for three cards: six distinct permutations.
 const permutations = new Set();
 for (let j2 = 0; j2 < 3; j2++) for (let j1 = 0; j1 < 2; j1++) {
  const choices = [(j2 + 0.5) / 3, (j1 + 0.5) / 2];
  let next = 0;
  permutations.add(shuffle([1, 2, 3], () => choices[next++]).join(','));
 }
 assert.equal(permutations.size, 6);
 console.log('All Chapter 13 checks passed.');
})();

## Explain without looking

Explain stability with two records that have the same score. What preprocessing cost is hidden by “sort, then binary search”? When would a bounded heap be preferable to sorting all results?

Adapted from the supplied Chapter 13 Sorting and Searching Algorithms guide. All 12 lesson tasks have worked solutions above. Random examples may vary; the verification cases use a deterministic generator.